# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

repo = "/content/Flyrank-assignment1"

if not os.path.exists(repo):
    !git clone https://github.com/vedant08mehta/Flyrank-assignment1.git

%cd /content/Flyrank-assignment1

print("Current directory:", os.getcwd())
print("Dataset exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 166 (delta 70), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 1.90 MiB | 16.24 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/Flyrank-assignment1
Current directory: /content/Flyrank-assignment1
Dataset exists: True


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue prioritizes pages that show stronger evidence of decline and greater potential impact. Each recommendation includes a reason code so that a human reviewer can understand why the page was prioritized instead of treating the model score as an automatic decision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Rank pages using the baseline action score created in Week 4.
# Higher score = higher priority for human review.
score_col = "action_score"

if score_col in df.columns:
    ranked = df.sort_values(score_col, ascending=False).copy()
else:
    # Recreate a simple priority score from available pre-prediction signals.
    ranked = df.copy()
    ranked["action_score"] = (
        ranked["impressions_90d"].rank(pct=True)
        + ranked["clicks_90d"].rank(pct=True)
        + ranked["avg_position"].rank(pct=True)
        - ranked["ctr"].rank(pct=True)
    )

ranked["reason_code"] = "HIGH_PRIORITY_REVIEW"

print("Top 10 recommended pages:")
display(
    ranked[
        ["content_id", "action_score", "reason_code"]
    ].head(10)
)

Top 10 recommended pages:


,content_id,action_score,reason_code
0,content_304f48230142,1.223783,HIGH_PRIORITY_REVIEW
1,content_a1fb4e703a9e,1.918417,HIGH_PRIORITY_REVIEW
2,content_9aa793d4d895,2.073983,HIGH_PRIORITY_REVIEW
3,content_331d6c4de07b,1.228250,HIGH_PRIORITY_REVIEW
4,content_d99b7a2d90ca,2.162150,HIGH_PRIORITY_REVIEW
5,content_d4084a4bc775,1.215350,HIGH_PRIORITY_REVIEW
6,content_9a34b442b552,0.466917,HIGH_PRIORITY_REVIEW
7,content_a63219c6e95a,1.385633,HIGH_PRIORITY_REVIEW
8,content_5e6c160719bc,2.274200,HIGH_PRIORITY_REVIEW
9,content_c27558df2b0c,0.719183,HIGH_PRIORITY_REVIEW


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The playbook is intended for content teams to use as decision-support when prioritizing pages for review, refresh, or further investigation. The recommendations should not be treated as automatic instructions to change or remove content. The model was evaluated on held-out client data, so performance may differ for new clients, changing traffic patterns, or data distributions outside the training and validation data.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Intended use ===")
print("Use: prioritize pages for human review.")
print("Action: investigate, refresh, or monitor selected pages.")
print("Not automatic: recommendations require human review.")
print("Limitation: performance may change for new clients or future data.")

=== Intended use ===
Use: prioritize pages for human review.
Action: investigate, refresh, or monitor selected pages.
Not automatic: recommendations require human review.
Limitation: performance may change for new clients or future data.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on a recommendation, a content specialist should review the page context, search intent, recent changes, business importance, and whether the observed decline is supported by enough data. Model recommendations should never automatically delete, rewrite, or publish content, and they should not override editorial, legal, brand, or business decisions. The model is a prioritization aid, not an autonomous content decision-maker.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
no_go_actions = [
    "Automatically delete content",
    "Automatically rewrite and publish content",
    "Automatically change business-critical pages",
    "Override editorial or legal review",
    "Treat the model score as causal proof"
]

print("=== No-go list ===")

for i, action in enumerate(no_go_actions, 1):
    print(f"{i}. {action}")

print("\nHuman review is required before taking action.")

=== No-go list ===
1. Automatically delete content
2. Automatically rewrite and publish content
3. Automatically change business-critical pages
4. Override editorial or legal review
5. Treat the model score as causal proof

Human review is required before taking action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be considered stale if Precision@50 falls materially from the validated 0.680 result, if the distribution of important features changes substantially, or if traffic and content patterns change enough that the original validation data no longer represents current conditions. These signals should trigger a review of the model and potentially a new training and validation cycle rather than silently continuing to use stale recommendations.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
validated_precision_at_50 = 0.680
current_target_precision_at_50 = 0.680

print("=== Monitoring check ===")
print(f"Validated Precision@50: {validated_precision_at_50:.3f}")
print(f"Current checked Precision@50: {current_target_precision_at_50:.3f}")

if current_target_precision_at_50 < validated_precision_at_50 - 0.10:
    print("TRIGGER: investigate model degradation and consider retraining.")
else:
    print("STATUS: no major Precision@50 degradation detected.")

print("\nRetrain/review triggers:")
print("- Material drop in Precision@50")
print("- Major feature-distribution shift")
print("- Major change in traffic/content patterns")
print("- Validation data no longer represents current use")

=== Monitoring check ===
Validated Precision@50: 0.680
Current checked Precision@50: 0.680
STATUS: no major Precision@50 degradation detected.

Retrain/review triggers:
- Material drop in Precision@50
- Major feature-distribution shift
- Major change in traffic/content patterns
- Validation data no longer represents current use


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The final action queue is exported as a CSV so that the ranked recommendations and their reason codes can be reused in the paper and reviewed independently of the notebook. The export is a decision-support artifact and does not represent automatic actions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "content_id",
    "action_score",
    "reason_code"
]

export_df = ranked[output_cols].head(50).copy()

output_path = "work/outputs/action_playbook.csv"
export_df.to_csv(output_path, index=False)

print(f"Exported: {output_path}")
print(f"Rows exported: {len(export_df)}")

display(export_df.head(10))

Exported: work/outputs/action_playbook.csv
Rows exported: 50


,content_id,action_score,reason_code
0,content_304f48230142,1.223783,HIGH_PRIORITY_REVIEW
1,content_a1fb4e703a9e,1.918417,HIGH_PRIORITY_REVIEW
2,content_9aa793d4d895,2.073983,HIGH_PRIORITY_REVIEW
3,content_331d6c4de07b,1.228250,HIGH_PRIORITY_REVIEW
4,content_d99b7a2d90ca,2.162150,HIGH_PRIORITY_REVIEW
5,content_d4084a4bc775,1.215350,HIGH_PRIORITY_REVIEW
6,content_9a34b442b552,0.466917,HIGH_PRIORITY_REVIEW
7,content_a63219c6e95a,1.385633,HIGH_PRIORITY_REVIEW
8,content_5e6c160719bc,2.274200,HIGH_PRIORITY_REVIEW
9,content_c27558df2b0c,0.719183,HIGH_PRIORITY_REVIEW


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.